In [1]:
# 01 — CLEANING SETUP & INPUT VALIDATION
# OFFLINE ONLY
# ============================================================

from pathlib import Path
import pandas as pd
import re
import html
import unicodedata

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

INPUT_FILE = (
    ROOT / "01_Raw Data"
    / "outputs" / "reports" / "batch7"
    / "manual_reconciliation"
    / "Batch7_Eligible_Tweets.csv"
)

OUTPUT_DIR = ROOT / "02_Cleaning" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert INPUT_FILE.exists(), (
    f"Input tidak ditemukan: {INPUT_FILE}"
)

df = pd.read_csv(
    INPUT_FILE,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

assert len(df) == 381, (
    f"Expected 381 tweet, ditemukan {len(df)}"
)

assert df["id"].is_unique, "Duplicate tweet ID."
assert df["text"].astype(str).str.strip().ne("").all()
assert df["final_inclusion_decision"].eq("INCLUDE").all()

print("CLEANING INPUT VALIDATION: PASS")
print("Eligible tweets:", len(df))
print("\nDistribution by case:")
display(df["case"].value_counts())


CLEANING INPUT VALIDATION: PASS
Eligible tweets: 381

Distribution by case:


case
sumatra_2025    149
palu_2018       118
lombok_2018     114
Name: count, dtype: int64

In [2]:
# 02 — INITIAL TEXT QUALITY AUDIT
# OFFLINE ONLY | NO TEXT MODIFICATION
# ============================================================

import pandas as pd
import re
import html
import unicodedata

assert len(df) == 381, "Input harus 381 tweet."
assert df["id"].is_unique, "Ada ID duplikat."

audit = df.copy()

# Simpan referensi teks asli tanpa perubahan
audit["text_original"] = audit["text"].astype("string")

# Pemeriksaan dasar
audit["text_length"] = audit["text_original"].str.len()
audit["is_empty"] = (
    audit["text_original"].str.strip().eq("")
)

audit["has_url"] = audit["text_original"].str.contains(
    r"https?://|www\.",
    regex=True,
    case=False,
    na=False
)

audit["has_mention"] = audit["text_original"].str.contains(
    r"(?<!\w)@\w+",
    regex=True,
    na=False
)

audit["has_hashtag"] = audit["text_original"].str.contains(
    r"(?<!\w)#\w+",
    regex=True,
    na=False
)

audit["has_html_entity"] = audit["text_original"].str.contains(
    r"&(?:amp|lt|gt|quot|apos|#\d+|#x[0-9a-f]+);",
    regex=True,
    case=False,
    na=False
)

# Duplikasi exact berdasarkan teks
audit["duplicate_exact_text"] = (
    audit.duplicated(
        subset=["text_original"],
        keep=False
    )
)

# Duplikasi setelah normalisasi ringan:
# Unicode NFKC, HTML unescape, lowercase, whitespace.
# Belum menghapus token ataupun mengubah teks original.
def comparison_key(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

audit["text_comparison_key"] = (
    audit["text_original"].map(comparison_key)
)

audit["duplicate_normalized_text"] = (
    audit.duplicated(
        subset=["text_comparison_key"],
        keep=False
    )
)

summary = pd.DataFrame([
    ["Total tweets", len(audit)],
    ["Unique tweet IDs", audit["id"].nunique()],
    ["Empty text", int(audit["is_empty"].sum())],
    ["Exact duplicate rows", int(audit["duplicate_exact_text"].sum())],
    ["Normalized duplicate rows", int(audit["duplicate_normalized_text"].sum())],
    ["Contains URL", int(audit["has_url"].sum())],
    ["Contains mention", int(audit["has_mention"].sum())],
    ["Contains hashtag", int(audit["has_hashtag"].sum())],
    ["Contains HTML entity", int(audit["has_html_entity"].sum())],
    ["Minimum text length", int(audit["text_length"].min())],
    ["Maximum text length", int(audit["text_length"].max())],
], columns=["Metric", "Value"])

print("=" * 60)
print("02 — INITIAL TEXT QUALITY AUDIT")
print("=" * 60)
display(summary)

print("\nExact duplicate samples:")
display(
    audit.loc[
        audit["duplicate_exact_text"],
        ["case", "id", "text_original"]
    ].head(15)
)

print("\nNormalized duplicate samples:")
display(
    audit.loc[
        audit["duplicate_normalized_text"],
        ["case", "id", "text_original"]
    ].head(15)
)

print("\nText length statistics:")
display(
    audit.groupby("case")["text_length"]
    .describe()
)

assert not audit["is_empty"].any(), (
    "Ada teks kosong yang perlu diperiksa."
)

print("\nINITIAL TEXT QUALITY AUDIT: PASS")
print("Original text: UNCHANGED")
print("No API call. No data deletion.")


02 — INITIAL TEXT QUALITY AUDIT


,Metric,Value
0,Total tweets,381
1,Unique tweet IDs,381
2,Empty text,0
3,Exact duplicate rows,7
4,Normalized duplicate rows,7
5,Contains URL,232
6,Contains mention,135
7,Contains hashtag,103
8,Contains HTML entity,24
9,Minimum text length,61



Exact duplicate samples:


,case,id,text_original
42,lombok_2018,1028971930978840577,Jika gempa Lombok ditetapkan sebagai bencana n...
43,lombok_2018,1028971930945298433,Jika gempa Lombok ditetapkan sebagai bencana n...
44,lombok_2018,1028968156587540480,Apalagi hingga kini pemerintah pusat belum men...
45,lombok_2018,1028968156130357248,Apalagi hingga kini pemerintah pusat belum men...
74,sumatra_2025,1996195214365757833,Bencana banjir serta longsor melanda sebagian ...
75,sumatra_2025,1996180113290088659,Bencana banjir serta longsor melanda sebagian ...
81,sumatra_2025,1996149775524376777,Bencana banjir serta longsor melanda sebagian ...



Normalized duplicate samples:


,case,id,text_original
42,lombok_2018,1028971930978840577,Jika gempa Lombok ditetapkan sebagai bencana n...
43,lombok_2018,1028971930945298433,Jika gempa Lombok ditetapkan sebagai bencana n...
44,lombok_2018,1028968156587540480,Apalagi hingga kini pemerintah pusat belum men...
45,lombok_2018,1028968156130357248,Apalagi hingga kini pemerintah pusat belum men...
74,sumatra_2025,1996195214365757833,Bencana banjir serta longsor melanda sebagian ...
75,sumatra_2025,1996180113290088659,Bencana banjir serta longsor melanda sebagian ...
81,sumatra_2025,1996149775524376777,Bencana banjir serta longsor melanda sebagian ...



Text length statistics:


,count,mean,std,min,25%,50%,75%,max
case,,,,,,,,
lombok_2018,114.0,164.508772,72.839409,61.0,107.25,142.5,231.5,301.0
palu_2018,118.0,176.567797,76.950587,73.0,98.0,164.0,252.5,362.0
sumatra_2025,149.0,474.550336,752.640509,73.0,197.0,282.0,399.0,6033.0



INITIAL TEXT QUALITY AUDIT: PASS
Original text: UNCHANGED
No API call. No data deletion.


In [3]:
# 03 — SAFE TEXT NORMALIZATION + LONG TEXT AUDIT
# OFFLINE ONLY | NO DATA DELETION
# ============================================================

import re
import html
import unicodedata
import pandas as pd

assert len(df) == 381
assert df["id"].is_unique
assert "text_original" in audit.columns

clean_df = audit.copy()

# Simpan teks asli dan metadata.
clean_df["text_original"] = (
    clean_df["text_original"].astype("string")
)

def normalize_basic_text(value):
    if pd.isna(value):
        return ""

    text = str(value)

    # Decode HTML entities berulang secara terbatas.
    # Misalnya &amp;amp; -> &
    for _ in range(3):
        decoded = html.unescape(text)
        if decoded == text:
            break
        text = decoded

    # Normalisasi Unicode.
    text = unicodedata.normalize("NFC", text)

    # Hilangkan karakter kontrol, kecuali whitespace
    # yang akan dinormalisasi di bawah.
    text = "".join(
        " " if (
            unicodedata.category(char) == "Cc"
        ) else char
        for char in text
    )

    # Hapus zero-width spacing yang sering tidak terlihat.
    text = re.sub(
        "[\u200b\u200c\u200d\ufeff]",
        "",
        text
    )

    # Rapikan seluruh whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Kolom hasil normalisasi, tanpa overwrite teks asli.
clean_df["text_normalized"] = (
    clean_df["text_original"]
    .map(normalize_basic_text)
    .astype("string")
)

# Audit perubahan.
clean_df["original_length"] = (
    clean_df["text_original"].str.len()
)
clean_df["normalized_length"] = (
    clean_df["text_normalized"].str.len()
)

clean_df["normalization_changed"] = (
    clean_df["text_original"]
    != clean_df["text_normalized"]
)

clean_df["length_difference"] = (
    clean_df["original_length"]
    - clean_df["normalized_length"]
)

# Flag teks panjang untuk pemeriksaan khusus.
LONG_TEXT_THRESHOLD = 1000

clean_df["is_long_text"] = (
    clean_df["original_length"]
    > LONG_TEXT_THRESHOLD
)

# Tidak boleh ada tweet hilang atau kosong.
assert len(clean_df) == len(df) == 381
assert clean_df["id"].is_unique
assert clean_df["text_normalized"].str.strip().ne("").all()
assert set(clean_df["id"]) == set(df["id"])

# Pastikan teks asli tidak berubah.
original_by_id = (
    audit.set_index("id")["text_original"]
    .astype("string")
)

current_by_id = (
    clean_df.set_index("id")["text_original"]
    .astype("string")
)

assert current_by_id.sort_index().equals(
    original_by_id.sort_index()
), "Teks asli berubah."

print("=" * 62)
print("03 — SAFE TEXT NORMALIZATION AUDIT")
print("=" * 62)
print("Total tweets       :", len(clean_df))
print(
    "Changed by normalization:",
    int(clean_df["normalization_changed"].sum())
)
print(
    "Unchanged              :",
    int((~clean_df["normalization_changed"]).sum())
)
print(
    "Long texts (>1000)     :",
    int(clean_df["is_long_text"].sum())
)
print(
    "Empty normalized texts :",
    int(clean_df["text_normalized"].str.strip().eq("").sum())
)

print("\nDistribution of long texts by case:")
display(
    clean_df.groupby("case")["is_long_text"]
    .agg(["sum", "count"])
)

print("\nExamples of changed text:")
display(
    clean_df.loc[
        clean_df["normalization_changed"],
        [
            "case",
            "id",
            "text_original",
            "text_normalized"
        ]
    ].head(10)
)

print("\nLong-text audit:")
display(
    clean_df.loc[
        clean_df["is_long_text"],
        [
            "case",
            "id",
            "original_length",
            "normalized_length",
            "text_original"
        ]
    ].sort_values(
        "original_length",
        ascending=False
    ).head(15)
)

print("\nCELL 03 NORMALIZATION: PASS")
print("Original data: PRESERVED")
print("Rows removed: 0")
print("No API calls.")


03 — SAFE TEXT NORMALIZATION AUDIT
Total tweets       : 381
Changed by normalization: 221
Unchanged              : 160
Long texts (>1000)     : 12
Empty normalized texts : 0

Distribution of long texts by case:


,sum,count
case,,
lombok_2018,0,114
palu_2018,0,118
sumatra_2025,12,149



Examples of changed text:


,case,id,text_original,text_normalized
7,lombok_2018,1026402242776117249,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...
8,lombok_2018,1026398913371074562,Bangda @Fahrihamzah Minta Status Gempa di Lomb...,Bangda @Fahrihamzah Minta Status Gempa di Lomb...
10,lombok_2018,1026372308284264448,Fahri Minta Status Gempa Lombok Jadi Bencana N...,Fahri Minta Status Gempa Lombok Jadi Bencana N...
11,palu_2018,1046068799915642880,@Jogja_Uncover ASTAGFIRULLOHALAZIM😭😭\nAmpuni k...,@Jogja_Uncover ASTAGFIRULLOHALAZIM😭😭 Ampuni ka...
13,palu_2018,1046044447295885312,@dapitdong Kami mendesak Palu - Donggala seger...,@dapitdong Kami mendesak Palu - Donggala seger...
14,palu_2018,1046042304354689027,@liputan6dotcom Kami mendesak Palu - Donggala ...,@liputan6dotcom Kami mendesak Palu - Donggala ...
15,palu_2018,1046031572699205633,Setelah Bencana Lombok ditolak menjadi Bencana...,Setelah Bencana Lombok ditolak menjadi Bencana...
18,palu_2018,1046016344519397376,Kami mendesak Palu - Donggala segera ditetapka...,Kami mendesak Palu - Donggala segera ditetapka...
20,palu_2018,1045955687405113344,Setuju... untuk mempercepat proses pertolongan...,Setuju... untuk mempercepat proses pertolongan...
21,palu_2018,1045880601759252482,Sudah selayaknya bencana gempa yg disertai tsu...,Sudah selayaknya bencana gempa yg disertai tsu...



Long-text audit:


,case,id,original_length,normalized_length,text_original
208,sumatra_2025,1995658121684152682,6033,5978,"Saat Aceh dan Sumatera ""Tenggelam"", Percakapan..."
147,sumatra_2025,1996389246203212130,5449,5418,Saya paham jika ada kepala daerah gelisah deng...
129,sumatra_2025,1996110111954329621,3475,3451,Ketika seorang ulama besar menyebut di depan r...
141,sumatra_2025,1996433134452375771,2385,2372,811 NYAWA DAN LUMPUR YANG TAK KUNJUNG KERING\n...
27,sumatra_2025,1994716450863579467,2132,2127,Bencana nasional yang terjadi di Sumatera bebe...
179,sumatra_2025,1995764391585919009,1354,1350,"Banjir dan tanah longsor yang melanda Aceh, Su..."
206,sumatra_2025,1995680229629776022,1345,1332,KLARIFIKASI HOAX - MISLEADING CONTENT \n[SALAH...
193,sumatra_2025,1995824414844485717,1253,1248,"Anggota Komisi V DPR RI Fraksi PKS, Abdul Hadi..."
112,sumatra_2025,1996059159654248664,1177,1172,"Bencana banjir bandang dan longsor di Aceh, Su..."
144,sumatra_2025,1996407464045211902,1127,1121,Presiden @prabowo diframing sedemikian oleh pe...



CELL 03 NORMALIZATION: PASS
Original data: PRESERVED
Rows removed: 0
No API calls.


In [4]:
# 04 — SOCIAL MEDIA TEXT CLEANING
# URL, MENTION, RT PREFIX, HASHTAG
# OFFLINE ONLY | ORIGINAL PRESERVED
# ============================================================

import re
import pandas as pd

assert len(clean_df) == 381
assert clean_df["id"].is_unique
assert "text_original" in clean_df.columns
assert "text_normalized" in clean_df.columns

social_df = clean_df.copy()

# ------------------------------------------------------------
# A. REGEX PATTERNS
# ------------------------------------------------------------

URL_PATTERN = re.compile(
    r"(?i)\b(?:https?://|www\.)[^\s<>]+"
)

MENTION_PATTERN = re.compile(
    r"(?<!\w)@[A-Za-z0-9_]{1,15}\b"
)

RT_PREFIX_PATTERN = re.compile(
    r"(?i)^\s*RT\s+"
)

HASHTAG_PATTERN = re.compile(
    r"(?<!\w)#([^\W_][\w]*)",
    flags=re.UNICODE
)

# ------------------------------------------------------------
# B. CLEANING FUNCTION
# ------------------------------------------------------------

def clean_social_text(value):
    if pd.isna(value):
        return ""

    text = str(value)

    # Hapus prefix retweet di awal teks.
    text = RT_PREFIX_PATTERN.sub("", text)

    # Hapus URL.
    text = URL_PATTERN.sub(" ", text)

    # Hapus mention, bukan konten kalimat.
    text = MENTION_PATTERN.sub(" ", text)

    # Pertahankan isi hashtag.
    text = HASHTAG_PATTERN.sub(r"\1", text)

    # Bersihkan separator retweet yang tertinggal
    # hanya bila berada di awal teks.
    text = re.sub(r"^\s*:\s*", "", text)

    # Rapikan spasi.
    text = re.sub(r"\s+", " ", text).strip()

    return text

# ------------------------------------------------------------
# C. APPLY CLEANING
# ------------------------------------------------------------

social_df["text_social_clean"] = (
    social_df["text_normalized"]
    .map(clean_social_text)
    .astype("string")
)

# ------------------------------------------------------------
# D. AUDIT CHANGES
# ------------------------------------------------------------

social_df["social_changed"] = (
    social_df["text_normalized"]
    != social_df["text_social_clean"]
)

social_df["social_length_before"] = (
    social_df["text_normalized"].str.len()
)

social_df["social_length_after"] = (
    social_df["text_social_clean"].str.len()
)

social_df["social_chars_removed"] = (
    social_df["social_length_before"]
    - social_df["social_length_after"]
)

social_df["social_empty"] = (
    social_df["text_social_clean"]
    .str.strip()
    .eq("")
)

# Flag kata penting terkait fokus penelitian.
POLICY_TERMS = [
    "bencana nasional",
    "status bencana",
    "penetapan",
    "pemerintah",
    "menolak",
    "setuju",
    "tidak",
    "bukan",
    "belum",
]

def present_terms(text):
    lowered = str(text).casefold()
    return [
        term for term in POLICY_TERMS
        if term in lowered
    ]

social_df["policy_terms_before"] = (
    social_df["text_normalized"]
    .map(present_terms)
)

social_df["policy_terms_after"] = (
    social_df["text_social_clean"]
    .map(present_terms)
)

social_df["policy_terms_lost"] = [
    sorted(set(before) - set(after))
    for before, after in zip(
        social_df["policy_terms_before"],
        social_df["policy_terms_after"]
    )
]

social_df["needs_semantic_review"] = (
    social_df["policy_terms_lost"].map(bool)
    | (
        social_df["social_chars_removed"]
        > social_df["social_length_before"] * 0.5
    )
)

# ------------------------------------------------------------
# E. ASSERTIONS
# ------------------------------------------------------------

assert len(social_df) == len(clean_df) == 381
assert social_df["id"].is_unique
assert set(social_df["id"]) == set(clean_df["id"])

assert not social_df["social_empty"].any(), (
    "Ada tweet menjadi kosong setelah social cleaning."
)

assert social_df["text_original"].equals(
    clean_df["text_original"]
), "Original text berubah."

assert social_df["text_normalized"].equals(
    clean_df["text_normalized"]
), "Normalized text berubah."

# ------------------------------------------------------------
# F. REPORT
# ------------------------------------------------------------

print("=" * 62)
print("04 — SOCIAL MEDIA TEXT CLEANING AUDIT")
print("=" * 62)

print("Total tweets              :", len(social_df))
print(
    "Changed text              :",
    int(social_df["social_changed"].sum())
)
print(
    "Unchanged text            :",
    int((~social_df["social_changed"]).sum())
)
print(
    "Empty after cleaning      :",
    int(social_df["social_empty"].sum())
)
print(
    "Need semantic review      :",
    int(social_df["needs_semantic_review"].sum())
)
print(
    "Lost policy keyword rows  :",
    int(
        social_df["policy_terms_lost"]
        .map(bool)
        .sum()
    )
)

print("\nExamples of changed text:")
display(
    social_df.loc[
        social_df["social_changed"],
        [
            "case",
            "id",
            "text_normalized",
            "text_social_clean",
        ]
    ].head(12)
)

print("\nFlagged for semantic review:")
display(
    social_df.loc[
        social_df["needs_semantic_review"],
        [
            "case",
            "id",
            "social_chars_removed",
            "policy_terms_lost",
            "text_normalized",
            "text_social_clean",
        ]
    ].head(20)
)

print("\nSOCIAL MEDIA CLEANING: PASS")
print("Original preserved: YES")
print("Rows deleted: 0")
print("No API calls.")


04 — SOCIAL MEDIA TEXT CLEANING AUDIT
Total tweets              : 381
Changed text              : 337
Unchanged text            : 44
Empty after cleaning      : 0
Need semantic review      : 5
Lost policy keyword rows  : 0

Examples of changed text:


,case,id,text_normalized,text_social_clean
0,palu_2018,1047490881878511616,@joxzin_jogja Yang mendukung status Bencana Na...,Yang mendukung status Bencana Nasional Lombok ...
1,lombok_2018,1026504085548101632,@LawanPoLitikJKW Kalo pemerintah punya sense o...,"Kalo pemerintah punya sense of crisis, mustiny..."
3,lombok_2018,1026424653298855937,BNPB: Tak Perlu Status Bencana Nasional dan Ba...,BNPB: Tak Perlu Status Bencana Nasional dan Ba...
4,lombok_2018,1026421529775493121,Pemerintah Diminta Naikan Status Lombok Jadi B...,Pemerintah Diminta Naikan Status Lombok Jadi B...
5,lombok_2018,1026411069466533888,Fahri Hamzah Ingin Gempa Lombok Ditetapkan seb...,Fahri Hamzah Ingin Gempa Lombok Ditetapkan seb...
6,lombok_2018,1026403261195071489,DPR Minta Gempa Lombok jadi Bencana Nasional #...,DPR Minta Gempa Lombok jadi Bencana Nasional a...
7,lombok_2018,1026402242776117249,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...
8,lombok_2018,1026398913371074562,Bangda @Fahrihamzah Minta Status Gempa di Lomb...,Bangda Minta Status Gempa di Lombok Jadi Benca...
9,lombok_2018,1026390946693271553,Gempa NTB Diminta jadi Bencana Nasional https:...,Gempa NTB Diminta jadi Bencana Nasional
10,lombok_2018,1026372308284264448,Fahri Minta Status Gempa Lombok Jadi Bencana N...,Fahri Minta Status Gempa Lombok Jadi Bencana N...



Flagged for semantic review:


,case,id,social_chars_removed,policy_terms_lost,text_normalized,text_social_clean
9,lombok_2018,1026390946693271553,48,[],Gempa NTB Diminta jadi Bencana Nasional https:...,Gempa NTB Diminta jadi Bencana Nasional
36,lombok_2018,1028974526841815041,56,[],Live! Gempa Lombok Bencana Nasional - @rakbir_...,Live! Gempa Lombok Bencana Nasional - bersama
87,palu_2018,1046820790853808128,144,[],Pemuda Muslimin Indonesia Harap Bencana Palu L...,Pemuda Muslimin Indonesia Harap Bencana Palu L...
282,lombok_2018,1028214950508916736,178,[],@Quvvatt @EV220622 @amrullah_sk @Reborn_Buroq ...,"Kalau lombok tidak dijadikan bencana nasional,..."
368,sumatra_2025,1995867615701676508,150,[],@Kimberley_PS08 @florieliciouss @are_inismynam...,"Naikan statusnya jadi bencana nasional, kita m..."



SOCIAL MEDIA CLEANING: PASS
Original preserved: YES
Rows deleted: 0
No API calls.


In [5]:
# 04A — SOCIAL CLEANING SEMANTIC REVIEW EXPORT
# OFFLINE ONLY | READ ONLY
# ============================================================

from pathlib import Path
import pandas as pd

assert len(social_df) == 381
assert social_df["id"].is_unique

AUDIT_DIR_04 = OUTPUT_DIR / "audits"
AUDIT_DIR_04.mkdir(parents=True, exist_ok=True)

REVIEW_PATH = (
    AUDIT_DIR_04 / "part4_semantic_review.xlsx"
)
FULL_AUDIT_PATH = (
    AUDIT_DIR_04 / "part4_full_audit.csv"
)

# Tidak menimpa file audit sebelumnya.
for path in [REVIEW_PATH, FULL_AUDIT_PATH]:
    if path.exists():
        raise FileExistsError(
            f"File sudah tersedia: {path}. "
            "Periksa atau arsipkan sebelum ekspor ulang."
        )

# A. Full audit seluruh tweet.
audit_cols = [
    "case",
    "id",
    "text_original",
    "text_normalized",
    "text_social_clean",
    "social_chars_removed",
    "needs_semantic_review",
]

full_audit = social_df[audit_cols].copy()
full_audit.to_csv(
    FULL_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# B. Audit khusus baris yang ditandai.
review = social_df.loc[
    social_df["needs_semantic_review"],
    audit_cols
].copy()

review["semantic_decision"] = ""
review["semantic_notes"] = ""
review["reviewer"] = ""
review["reviewed_at"] = ""

# C. Ringkasan.
summary = pd.DataFrame([
    ["Total eligible tweets", len(social_df)],
    ["Changed by social cleaning",
     int(social_df["social_changed"].sum())],
    ["Semantic review required", len(review)],
    ["Empty after cleaning",
     int(social_df["social_empty"].sum())],
], columns=["Metric", "Value"])

# Proteksi nilai teks agar tidak dieksekusi
# sebagai formula ketika dibuka di Excel.
def safe_excel(frame):
    result = frame.copy()
    for column in result.columns:
        result[column] = result[column].map(
            lambda value: "'" + value
            if isinstance(value, str)
            and value.startswith(("=", "+", "-", "@"))
            else value
        )
    return result

with pd.ExcelWriter(
    REVIEW_PATH,
    engine="openpyxl"
) as writer:
    safe_excel(summary).to_excel(
        writer, sheet_name="SUMMARY", index=False
    )

    safe_excel(review).to_excel(
        writer, sheet_name="SEMANTIC_REVIEW",
        index=False
    )

    ws = writer.sheets["SEMANTIC_REVIEW"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    from openpyxl.styles import Alignment, Font
    from openpyxl.utils import get_column_letter

    for column_index, name in enumerate(
        review.columns, start=1
    ):
        letter = get_column_letter(column_index)
        ws.column_dimensions[letter].width = (
            75 if name.startswith("text_") else 28
        )

        if name == "id":
            for row in range(2, ws.max_row + 1):
                cell = ws.cell(row, column_index)
                cell.number_format = "@"

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True)

print("=" * 60)
print("04A — SEMANTIC REVIEW EXPORT")
print("=" * 60)
print("All tweets       :", len(social_df))
print("Flagged for review:", len(review))
print("Review Excel     :", REVIEW_PATH)
print("Full audit CSV   :", FULL_AUDIT_PATH)
print("Rows deleted     : 0")
print("Source modified  : NO")
print("=" * 60)


04A — SEMANTIC REVIEW EXPORT
All tweets       : 381
Flagged for review: 5
Review Excel     : C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part4_semantic_review.xlsx
Full audit CSV   : C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part4_full_audit.csv
Rows deleted     : 0
Source modified  : NO


In [7]:
# 04B — SEMANTIC REVIEW RECONCILIATION
# OFFLINE ONLY | NO SOURCE MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd

assert len(social_df) == 381
assert social_df["id"].is_unique

review_path = (
    OUTPUT_DIR / "audits" / "part4_semantic_review.xlsx"
)

assert review_path.is_file(), f"File tidak ada: {review_path}"

review = pd.read_excel(
    review_path,
    sheet_name="SEMANTIC_REVIEW",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

required = [
    "case", "id", "semantic_decision",
    "semantic_notes", "reviewer", "reviewed_at"
]

missing = set(required) - set(review.columns)
assert not missing, f"Kolom hilang: {missing}"

review["id"] = review["id"].astype("string").str.strip()
review["semantic_decision"] = (
    review["semantic_decision"]
    .astype("string").str.strip().str.upper()
)

# Validasi terhadap lima ID yang ditandai Cell 04
expected = social_df.loc[
    social_df["needs_semantic_review"],
    ["case", "id"]
].copy()

expected["id"] = expected["id"].astype("string")

assert len(review) == len(expected) == 5
assert review["id"].is_unique
assert set(review["id"]) == set(expected["id"])

review_pairs = set(zip(review["case"], review["id"]))
expected_pairs = set(zip(expected["case"], expected["id"]))
assert review_pairs == expected_pairs, (
    "Pasangan case dan ID tidak cocok."
)

allowed = {"ACCEPT", "RESTORE"}
invalid = review.loc[
    ~review["semantic_decision"].isin(allowed),
    ["id", "semantic_decision"]
]

if not invalid.empty:
    display(invalid)
    raise ValueError(
        "Masih ada keputusan kosong/REVIEW/tidak valid."
    )

for col in ["semantic_notes", "reviewer", "reviewed_at"]:
    blank = review[col].astype("string").str.strip().eq("")
    if blank.any():
        display(review.loc[blank, ["id", col]])
        raise ValueError(f"Ada isian kosong: {col}")

# Jangan ubah data hasil Cell 04 secara langsung.
verified_df = social_df.copy()
verified_df["id"] = verified_df["id"].astype("string")

decisions = review.set_index("id")["semantic_decision"]
verified_df["semantic_decision"] = (
    verified_df["id"].map(decisions).fillna("NOT_FLAGGED")
)

verified_df["text_semantic_verified"] = (
    verified_df["text_social_clean"].copy()
)

# Untuk RESTORE gunakan teks sebelum social cleaning.
# Cara ini konservatif dan dapat diaudit, meskipun URL
# dan mention yang semula ada juga akan dipertahankan.
restore_mask = verified_df["semantic_decision"].eq("RESTORE")

verified_df.loc[
    restore_mask, "text_semantic_verified"
] = verified_df.loc[
    restore_mask, "text_normalized"
]

verified_df["semantic_review_notes"] = (
    verified_df["id"].map(
        review.set_index("id")["semantic_notes"]
    ).fillna("")
)

verified_df["semantic_reviewer"] = (
    verified_df["id"].map(
        review.set_index("id")["reviewer"]
    ).fillna("")
)

# Integritas data
assert len(verified_df) == 381
assert verified_df["id"].is_unique
assert verified_df["text_semantic_verified"].astype(
    "string"
).str.strip().ne("").all()

assert verified_df["text_original"].equals(
    social_df["text_original"]
)

assert verified_df["text_normalized"].equals(
    social_df["text_normalized"]
)

print("=" * 60)
print("04B — SEMANTIC REVIEW RECONCILIATION")
print("=" * 60)
print("Total tweets :", len(verified_df))
print("Flagged      :", len(review))
print("ACCEPT       :", int((review["semantic_decision"] == "ACCEPT").sum()))
print("RESTORE      :", int((review["semantic_decision"] == "RESTORE").sum()))
print("Unresolved  :", len(invalid))
print("Original preserved: YES")
print("=" * 60)

display(
    verified_df.loc[
        verified_df["semantic_decision"].isin(
            ["ACCEPT", "RESTORE"]
        ),
        [
            "case", "id", "semantic_decision",
            "text_normalized",
            "text_social_clean",
            "text_semantic_verified",
            "semantic_review_notes"
        ]
    ]
)

print("SEMANTIC RECONCILIATION: PASS")


04B — SEMANTIC REVIEW RECONCILIATION
Total tweets : 381
Flagged      : 5
ACCEPT       : 5
RESTORE      : 0
Unresolved  : 0
Original preserved: YES


,case,id,semantic_decision,text_normalized,text_social_clean,text_semantic_verified,semantic_review_notes
9,lombok_2018,1026390946693271553,ACCEPT,Gempa NTB Diminta jadi Bencana Nasional https:...,Gempa NTB Diminta jadi Bencana Nasional,Gempa NTB Diminta jadi Bencana Nasional,maksudnya sudah jelas
36,lombok_2018,1028974526841815041,ACCEPT,Live! Gempa Lombok Bencana Nasional - @rakbir_...,Live! Gempa Lombok Bencana Nasional - bersama,Live! Gempa Lombok Bencana Nasional - bersama,membahas penteapan bencana nasional lewat live
87,palu_2018,1046820790853808128,ACCEPT,Pemuda Muslimin Indonesia Harap Bencana Palu L...,Pemuda Muslimin Indonesia Harap Bencana Palu L...,Pemuda Muslimin Indonesia Harap Bencana Palu L...,maksudnya sudah jelas
282,lombok_2018,1028214950508916736,ACCEPT,@Quvvatt @EV220622 @amrullah_sk @Reborn_Buroq ...,"Kalau lombok tidak dijadikan bencana nasional,...","Kalau lombok tidak dijadikan bencana nasional,...",maksudnya sudah jelas
368,sumatra_2025,1995867615701676508,ACCEPT,@Kimberley_PS08 @florieliciouss @are_inismynam...,"Naikan statusnya jadi bencana nasional, kita m...","Naikan statusnya jadi bencana nasional, kita m...",maksudnya sudah jelas


SEMANTIC RECONCILIATION: PASS


In [8]:
# 05 — CONSERVATIVE LINGUISTIC NORMALIZATION
# OFFLINE ONLY | NO ROW DELETION
# ============================================================

import re
import unicodedata
import pandas as pd

assert len(verified_df) == 381
assert verified_df["id"].is_unique
assert "text_semantic_verified" in verified_df.columns

ling_df = verified_df.copy()

# A. Istilah penting yang harus dipertahankan.
NEGATION_TERMS = {
    "tidak", "bukan", "belum", "jangan",
    "tak", "gak", "nggak", "enggak",
    "ga", "gk", "tdk", "bkn", "blm"
}

POLICY_PHRASES = [
    "bencana nasional",
    "status bencana",
    "penetapan",
    "pemerintah",
    "menolak",
    "setuju",
]

# B. Normalisasi konservatif.
def normalize_linguistic(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Lowercase untuk representasi pemrosesan.
    text = text.casefold()

    # Unicode NFC.
    text = unicodedata.normalize("NFC", text)

    # Rapikan whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    # Pertahankan tanda baca dan emoji.
    # Jangan lakukan stemming atau stopword removal.
    return text

ling_df["text_linguistic"] = (
    ling_df["text_semantic_verified"]
    .map(normalize_linguistic)
    .astype("string")
)

# C. Perlindungan istilah penting.
def find_negations(text):
    tokens = re.findall(r"\b\w+\b", str(text).casefold())
    return sorted(
        set(tokens) & NEGATION_TERMS
    )

def find_policy_phrases(text):
    text = str(text).casefold()
    return sorted(
        phrase for phrase in POLICY_PHRASES
        if phrase in text
    )

ling_df["negation_before"] = (
    ling_df["text_semantic_verified"]
    .map(find_negations)
)

ling_df["negation_after"] = (
    ling_df["text_linguistic"]
    .map(find_negations)
)

ling_df["negation_lost"] = [
    sorted(set(before) - set(after))
    for before, after in zip(
        ling_df["negation_before"],
        ling_df["negation_after"]
    )
]

ling_df["policy_before"] = (
    ling_df["text_semantic_verified"]
    .map(find_policy_phrases)
)

ling_df["policy_after"] = (
    ling_df["text_linguistic"]
    .map(find_policy_phrases)
)

ling_df["policy_lost"] = [
    sorted(set(before) - set(after))
    for before, after in zip(
        ling_df["policy_before"],
        ling_df["policy_after"]
    )
]

# D. Audit potensi duplikasi teks.
ling_df["duplicate_linguistic"] = (
    ling_df.duplicated(
        subset=["text_linguistic"],
        keep=False
    )
)

ling_df["needs_linguistic_review"] = (
    ling_df["negation_lost"].map(bool)
    | ling_df["policy_lost"].map(bool)
)

# E. Assertions.
assert len(ling_df) == 381
assert ling_df["id"].is_unique
assert ling_df["text_linguistic"].str.strip().ne("").all()

assert ling_df["text_original"].equals(
    verified_df["text_original"]
)

assert ling_df["text_semantic_verified"].equals(
    verified_df["text_semantic_verified"]
)

assert not ling_df["needs_linguistic_review"].any(), (
    "Ada istilah penting yang hilang."
)

# F. Summary.
print("=" * 62)
print("05 — CONSERVATIVE LINGUISTIC NORMALIZATION")
print("=" * 62)

print("Total tweets             :", len(ling_df))
print(
    "Changed text             :",
    int(
        (
            ling_df["text_semantic_verified"]
            != ling_df["text_linguistic"]
        ).sum()
    )
)
print(
    "Lost negation rows       :",
    int(ling_df["negation_lost"].map(bool).sum())
)
print(
    "Lost policy phrase rows  :",
    int(ling_df["policy_lost"].map(bool).sum())
)
print(
    "Duplicate text rows      :",
    int(ling_df["duplicate_linguistic"].sum())
)
print(
    "Need linguistic review   :",
    int(ling_df["needs_linguistic_review"].sum())
)

print("\nExamples:")
display(
    ling_df[
        [
            "case",
            "id",
            "text_semantic_verified",
            "text_linguistic"
        ]
    ].head(10)
)

print("\nDuplicate samples:")
display(
    ling_df.loc[
        ling_df["duplicate_linguistic"],
        [
            "case", "id",
            "text_linguistic"
        ]
    ].head(15)
)

print("\nCELL 05 LINGUISTIC NORMALIZATION: PASS")
print("Original text preserved.")
print("No data deletion. No API calls.")


05 — CONSERVATIVE LINGUISTIC NORMALIZATION
Total tweets             : 381
Changed text             : 372
Lost negation rows       : 0
Lost policy phrase rows  : 0
Duplicate text rows      : 54
Need linguistic review   : 0

Examples:


,case,id,text_semantic_verified,text_linguistic
0,palu_2018,1047490881878511616,Yang mendukung status Bencana Nasional Lombok ...,yang mendukung status bencana nasional lombok ...
1,lombok_2018,1026504085548101632,"Kalo pemerintah punya sense of crisis, mustiny...","kalo pemerintah punya sense of crisis, mustiny..."
2,lombok_2018,1026427713525927938,Gempa Lombok harus ditetapkan status “Bencana ...,gempa lombok harus ditetapkan status “bencana ...
3,lombok_2018,1026424653298855937,BNPB: Tak Perlu Status Bencana Nasional dan Ba...,bnpb: tak perlu status bencana nasional dan ba...
4,lombok_2018,1026421529775493121,Pemerintah Diminta Naikan Status Lombok Jadi B...,pemerintah diminta naikan status lombok jadi b...
5,lombok_2018,1026411069466533888,Fahri Hamzah Ingin Gempa Lombok Ditetapkan seb...,fahri hamzah ingin gempa lombok ditetapkan seb...
6,lombok_2018,1026403261195071489,DPR Minta Gempa Lombok jadi Bencana Nasional a...,dpr minta gempa lombok jadi bencana nasional a...
7,lombok_2018,1026402242776117249,Wakil Ketua DPR RI Fahri Hamzah meminta pemeri...,wakil ketua dpr ri fahri hamzah meminta pemeri...
8,lombok_2018,1026398913371074562,Bangda Minta Status Gempa di Lombok Jadi Benca...,bangda minta status gempa di lombok jadi benca...
9,lombok_2018,1026390946693271553,Gempa NTB Diminta jadi Bencana Nasional,gempa ntb diminta jadi bencana nasional



Duplicate samples:


,case,id,text_linguistic
13,palu_2018,1046044447295885312,kami mendesak palu - donggala segera ditetapka...
14,palu_2018,1046042304354689027,kami mendesak palu - donggala segera ditetapka...
18,palu_2018,1046016344519397376,kami mendesak palu - donggala segera ditetapka...
37,lombok_2018,1028974452200169474,"saksikan ""gempa lombok bencana nasional"". live..."
38,lombok_2018,1028974451830939648,"saksikan ""gempa lombok bencana nasional"". live..."
39,lombok_2018,1028973189748219911,. akan mengajak anda mendengar secara langsung...
40,lombok_2018,1028973189085519872,. akan mengajak anda mendengar secara langsung...
41,lombok_2018,1028973189039177728,. akan mengajak anda mendengar secara langsung...
42,lombok_2018,1028971930978840577,jika gempa lombok ditetapkan sebagai bencana n...
43,lombok_2018,1028971930945298433,jika gempa lombok ditetapkan sebagai bencana n...



CELL 05 LINGUISTIC NORMALIZATION: PASS
Original text preserved.
No data deletion. No API calls.


In [9]:
# 06 — DUPLICATE & SEMANTIC SAFEGUARD AUDIT
# OFFLINE ONLY | NO DATA DELETION
# ============================================================

import hashlib
import pandas as pd

assert len(ling_df) == 381
assert ling_df["id"].is_unique
assert ling_df["text_linguistic"].str.strip().ne("").all()

dup_df = ling_df.copy()

# A. Kunci teks identik dari hasil cleaning.
def make_text_hash(text):
    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()

dup_df["text_hash"] = (
    dup_df["text_linguistic"].map(make_text_hash)
)

# Grup berdasarkan teks sama pada kasus yang sama.
# Hash SHA256 digunakan untuk ID grup yang stabil.
dup_df["duplicate_group_id"] = [
    hashlib.sha256(
        (str(case) + "\x00" + str(text)).encode("utf-8")
    ).hexdigest()[:16]
    for case, text in zip(
        dup_df["case"],
        dup_df["text_linguistic"]
    )
]

# B. Ukuran grup duplikasi.
dup_df["duplicate_group_size"] = (
    dup_df.groupby("duplicate_group_id")["id"]
    .transform("size")
)

dup_df["is_duplicate_in_case"] = (
    dup_df["duplicate_group_size"] > 1
)

# Deteksi teks yang muncul di lebih dari satu kasus.
dup_df["cross_case_count"] = (
    dup_df.groupby("text_hash")["case"]
    .transform("nunique")
)

dup_df["is_cross_case_duplicate"] = (
    dup_df["cross_case_count"] > 1
)

# C. Kelompok duplikat untuk audit.
duplicate_groups = (
    dup_df.loc[
        dup_df["is_duplicate_in_case"]
    ]
    .groupby(
        ["case", "duplicate_group_id"],
        as_index=False
    )
    .agg(
        group_size=("id", "size"),
        sample_text=("text_linguistic", "first")
    )
    .sort_values(
        ["group_size", "case"],
        ascending=[False, True]
    )
)

# D. Tandai semua baris: tidak menghapus data.
dup_df["duplicate_review_status"] = (
    dup_df["is_duplicate_in_case"].map(
        {True: "PENDING_GROUP_REVIEW",
         False: "NOT_DUPLICATE"}
    )
)

# E. Integritas data.
assert len(dup_df) == len(ling_df) == 381
assert dup_df["id"].is_unique
assert set(dup_df["id"]) == set(ling_df["id"])

assert dup_df["text_original"].equals(
    ling_df["text_original"]
)

assert dup_df["text_linguistic"].equals(
    ling_df["text_linguistic"]
)

# F. Laporan per kasus.
case_summary = (
    dup_df.groupby("case", as_index=False)
    .agg(
        total_tweets=("id", "size"),
        duplicate_rows=(
            "is_duplicate_in_case", "sum"
        ),
        cross_case_rows=(
            "is_cross_case_duplicate", "sum"
        )
    )
)

# G. Export audit, tanpa menimpa hasil lama.
AUDIT_DIR_06 = OUTPUT_DIR / "audits"
AUDIT_DIR_06.mkdir(parents=True, exist_ok=True)

details_path = (
    AUDIT_DIR_06 / "part6_duplicate_details.csv"
)

groups_path = (
    AUDIT_DIR_06 / "part6_duplicate_groups.csv"
)

summary_path = (
    AUDIT_DIR_06 / "part6_duplicate_by_case.csv"
)

for path in [
    details_path,
    groups_path,
    summary_path
]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah ada: {path}"
        )

detail_cols = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "duplicate_group_id",
    "duplicate_group_size",
    "is_duplicate_in_case",
    "is_cross_case_duplicate",
    "duplicate_review_status"
]

dup_df[detail_cols].to_csv(
    details_path,
    index=False,
    encoding="utf-8-sig"
)

duplicate_groups.to_csv(
    groups_path,
    index=False,
    encoding="utf-8-sig"
)

case_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

# H. Report.
print("=" * 65)
print("06 — DUPLICATE & SEMANTIC SAFEGUARD AUDIT")
print("=" * 65)

print("Total tweets           :", len(dup_df))
print("Unique tweet IDs       :", dup_df["id"].nunique())
print(
    "Duplicate rows in case:",
    int(dup_df["is_duplicate_in_case"].sum())
)
print(
    "Duplicate groups      :",
    len(duplicate_groups)
)
print(
    "Cross-case rows       :",
    int(dup_df["is_cross_case_duplicate"].sum())
)
print("Deleted rows          : 0")

print("\nDuplicate summary by case:")
display(case_summary)

print("\nLargest duplicate groups:")
display(duplicate_groups.head(20))

print("\nAudit files:")
print(details_path)
print(groups_path)
print(summary_path)

print("\nCELL 06 DUPLICATE AUDIT: PASS")


06 — DUPLICATE & SEMANTIC SAFEGUARD AUDIT
Total tweets           : 381
Unique tweet IDs       : 381
Duplicate rows in case: 54
Duplicate groups      : 21
Cross-case rows       : 0
Deleted rows          : 0

Duplicate summary by case:


,case,total_tweets,duplicate_rows,cross_case_rows
0,lombok_2018,114,22,0
1,palu_2018,118,22,0
2,sumatra_2025,149,10,0



Largest duplicate groups:


,case,duplicate_group_id,group_size,sample_text
12,palu_2018,9abf3a2407b60477,5,menteri luhut: penanganan gempa palu lebih dar...
19,sumatra_2025,938dc3441b24d7e6,4,banjir bandang dan tanah longsor berskala masi...
20,sumatra_2025,e8bd5577ee8847ae,4,bencana banjir serta longsor melanda sebagian ...
4,lombok_2018,d1c867fda398d1f5,3,pemerintah putuskan gempa lombok bukan bencana...
6,lombok_2018,e614904bd83e68c4,3,. akan mengajak anda mendengar secara langsung...
7,lombok_2018,f38f0b7eb78e3bc8,3,ribuan bangunan hancur dan ratusan orang tewas...
8,lombok_2018,fe41e5156d710ad9,3,alasan jokowi belum tetapkan bencana nasional ...
13,palu_2018,b481f1da19b127e1,3,kami mendesak palu - donggala segera ditetapka...
0,lombok_2018,2d9c177232fc1d3b,2,"saksikan ""gempa lombok bencana nasional"". live..."
1,lombok_2018,55d6861ce57d6a53,2,apalagi hingga kini pemerintah pusat belum men...



Audit files:
C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part6_duplicate_details.csv
C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part6_duplicate_groups.csv
C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part6_duplicate_by_case.csv

CELL 06 DUPLICATE AUDIT: PASS


In [10]:
# 07 — DUPLICATE & LONG TEXT REVIEW PREPARATION
# OFFLINE ONLY | NO DATA DELETION
# ============================================================

from pathlib import Path
import pandas as pd
import json

assert len(dup_df) == 381
assert dup_df["id"].is_unique

REVIEW_DIR_07 = OUTPUT_DIR / "audits"
REVIEW_DIR_07.mkdir(parents=True, exist_ok=True)

REVIEW_FILE_07 = (
    REVIEW_DIR_07 / "part7_content_review.xlsx"
)

SUMMARY_FILE_07 = (
    REVIEW_DIR_07 / "part7_content_review_summary.json"
)

for path in [REVIEW_FILE_07, SUMMARY_FILE_07]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah ada: {path}"
        )

# ------------------------------------------------------------
# A. Duplicate review — seluruh anggota 21 grup
# ------------------------------------------------------------

duplicate_review = dup_df.loc[
    dup_df["is_duplicate_in_case"]
].copy()

duplicate_review = duplicate_review[
    [
        "case",
        "id",
        "duplicate_group_id",
        "duplicate_group_size",
        "text_original",
        "text_semantic_verified",
        "text_linguistic"
    ]
].sort_values(
    ["case", "duplicate_group_id", "id"]
)

duplicate_review["review_decision"] = ""
duplicate_review["review_notes"] = ""

# ------------------------------------------------------------
# B. Long text review — berdasarkan teks asli
# ------------------------------------------------------------

long_review = dup_df.loc[
    dup_df["text_original"].str.len() > 1000
].copy()

long_review["original_length"] = (
    long_review["text_original"].str.len()
)

long_review = long_review[
    [
        "case",
        "id",
        "original_length",
        "text_original",
        "text_semantic_verified",
        "text_linguistic"
    ]
].sort_values(
    "original_length",
    ascending=False
)

long_review["review_decision"] = ""
long_review["review_notes"] = ""

# ------------------------------------------------------------
# C. Validation
# ------------------------------------------------------------

assert len(duplicate_review) == 54, (
    f"Expected 54 duplicate rows; got {len(duplicate_review)}"
)

assert duplicate_review["duplicate_group_id"].nunique() == 21

assert len(long_review) == 12, (
    f"Expected 12 long texts; got {len(long_review)}"
)

assert set(long_review["case"]) == {"sumatra_2025"}

# ------------------------------------------------------------
# D. Instructions
# ------------------------------------------------------------

instructions = pd.DataFrame([
    {
        "Sheet": "DUPLICATE_REVIEW",
        "Decision": "KEEP_ALL",
        "Meaning": "Pertahankan semua tweet sebagai postingan berbeda."
    },
    {
        "Sheet": "DUPLICATE_REVIEW",
        "Decision": "ONE_FOR_MODEL",
        "Meaning": "Gunakan satu perwakilan grup untuk modeling; "
                   "semua tweet tetap disimpan."
    },
    {
        "Sheet": "DUPLICATE_REVIEW",
        "Decision": "REVIEW",
        "Meaning": "Memerlukan pemeriksaan tambahan."
    },
    {
        "Sheet": "LONG_TEXT_REVIEW",
        "Decision": "KEEP_FULL",
        "Meaning": "Pertahankan teks panjang seluruhnya."
    },
    {
        "Sheet": "LONG_TEXT_REVIEW",
        "Decision": "REVIEW",
        "Meaning": "Memerlukan pemeriksaan isi lebih lanjut."
    }
])

# ------------------------------------------------------------
# E. Export to Excel
# ------------------------------------------------------------

with pd.ExcelWriter(
    REVIEW_FILE_07,
    engine="openpyxl"
) as writer:

    instructions.to_excel(
        writer,
        sheet_name="README",
        index=False
    )

    duplicate_review.to_excel(
        writer,
        sheet_name="DUPLICATE_REVIEW",
        index=False
    )

    long_review.to_excel(
        writer,
        sheet_name="LONG_TEXT_REVIEW",
        index=False
    )

    from openpyxl.styles import Font, Alignment

    for ws in writer.sheets.values():
        ws.freeze_panes = "C2"
        ws.auto_filter.ref = ws.dimensions

        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(
                wrap_text=True
            )

        for column in ws.columns:
            header = str(column[0].value)

            if header.startswith("text_"):
                ws.column_dimensions[
                    column[0].column_letter
                ].width = 75
            else:
                ws.column_dimensions[
                    column[0].column_letter
                ].width = 27

            # Semua ID disimpan sebagai teks.
            if header == "id":
                for cell in column[1:]:
                    if cell.value is not None:
                        cell.value = str(cell.value)
                        cell.number_format = "@"

            # Amankan nilai string agar Excel
            # tidak menafsirkannya sebagai formula.
            if header.startswith("text_"):
                for cell in column[1:]:
                    if isinstance(cell.value, str):
                        cell.data_type = "s"

# ------------------------------------------------------------
# F. Summary JSON
# ------------------------------------------------------------

summary_07 = {
    "total_tweets": 381,
    "duplicate_rows": len(duplicate_review),
    "duplicate_groups": int(
        duplicate_review["duplicate_group_id"].nunique()
    ),
    "long_text_rows": len(long_review),
    "cross_case_duplicate_rows": int(
        dup_df["is_cross_case_duplicate"].sum()
    ),
    "rows_deleted": 0,
    "status": "REVIEW_PENDING"
}

with open(
    SUMMARY_FILE_07,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary_07,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 64)
print("07 — CONTENT REVIEW PREPARATION")
print("=" * 64)

for key, value in summary_07.items():
    print(f"{key}: {value}")

print("\nWorkbook:", REVIEW_FILE_07)
print("Summary :", SUMMARY_FILE_07)

print("\nCELL 07 EXPORT: PASS")
print("Manual review status: PENDING")
print("No rows deleted.")


07 — CONTENT REVIEW PREPARATION
total_tweets: 381
duplicate_rows: 54
duplicate_groups: 21
long_text_rows: 12
cross_case_duplicate_rows: 0
rows_deleted: 0
status: REVIEW_PENDING

Workbook: C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part7_content_review.xlsx
Summary : C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\audits\part7_content_review_summary.json

CELL 07 EXPORT: PASS
Manual review status: PENDING
No rows deleted.


In [11]:
# 08 — MANUAL CONTENT REVIEW RECONCILIATION
# ONE REPRESENTATIVE PER DUPLICATE GROUP
# OFFLINE ONLY | NO SOURCE MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import json

assert len(dup_df) == 381
assert dup_df["id"].astype(str).is_unique

review_file = (
    OUTPUT_DIR / "audits" / "part7_content_review.xlsx"
)

assert review_file.exists(), f"Workbook tidak ditemukan: {review_file}"

# ------------------------------------------------------------
# A. LOAD MANUAL REVIEW
# ------------------------------------------------------------

reviews = pd.read_excel(
    review_file,
    sheet_name=None,
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

duplicate_review = reviews["DUPLICATE_REVIEW"].copy()
long_review = reviews["LONG_TEXT_REVIEW"].copy()

for frame in [duplicate_review, long_review]:
    frame["id"] = frame["id"].astype("string").str.strip()
    frame["case"] = frame["case"].astype("string").str.strip()
    frame["review_decision"] = (
        frame["review_decision"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

assert len(duplicate_review) == 54
assert len(long_review) == 12

assert duplicate_review["id"].is_unique
assert long_review["id"].is_unique

# ------------------------------------------------------------
# B. VALIDATE WORKBOOK AGAINST PIPELINE
# ------------------------------------------------------------

expected_dups = dup_df.loc[
    dup_df["is_duplicate_in_case"],
    ["case", "id", "duplicate_group_id"]
].copy()

expected_long = dup_df.loc[
    dup_df["text_original"].str.len() > 1000,
    ["case", "id"]
].copy()

for frame in [expected_dups, expected_long]:
    frame["id"] = frame["id"].astype("string")

assert set(zip(
    duplicate_review["case"],
    duplicate_review["id"]
)) == set(zip(
    expected_dups["case"],
    expected_dups["id"]
)), "Duplicate review IDs/cases tidak sesuai."

assert set(zip(
    long_review["case"],
    long_review["id"]
)) == set(zip(
    expected_long["case"],
    expected_long["id"]
)), "Long-text review IDs/cases tidak sesuai."

expected_group_map = (
    expected_dups.set_index("id")["duplicate_group_id"]
)

actual_group_map = (
    duplicate_review.set_index("id")["duplicate_group_id"]
)

assert actual_group_map.sort_index().equals(
    expected_group_map.sort_index()
), "Duplicate group ID tidak sesuai."

# Pastikan teks dalam workbook tidak diubah.
for sheet_df in [duplicate_review, long_review]:
    for text_col in [
        "text_original",
        "text_semantic_verified",
        "text_linguistic"
    ]:
        expected_text = (
            dup_df.assign(
                id=dup_df["id"].astype("string")
            ).set_index("id")[text_col]
        )

        actual_text = sheet_df.set_index("id")[text_col]

        assert all(
            str(actual_text.loc[tweet_id])
            == str(expected_text.loc[tweet_id])
            for tweet_id in actual_text.index
        ), f"Teks workbook berubah: {text_col}"

# ------------------------------------------------------------
# C. VALIDATE REPRESENTATIVE SELECTION
# ------------------------------------------------------------

allowed_duplicate = {"ONE_FOR_MODEL", ""}

assert set(
    duplicate_review["review_decision"]
).issubset(allowed_duplicate), (
    "Ada keputusan duplikat yang tidak dikenali."
)

selected_review = duplicate_review.loc[
    duplicate_review["review_decision"]
    .eq("ONE_FOR_MODEL")
].copy()

selection_counts = (
    selected_review.groupby("duplicate_group_id")
    .size()
)

all_duplicate_groups = set(
    duplicate_review["duplicate_group_id"]
)

assert set(selection_counts.index) == all_duplicate_groups, (
    "Ada grup duplikat tanpa perwakilan."
)

assert selection_counts.eq(1).all(), (
    "Setiap grup harus memiliki tepat satu perwakilan."
)

assert len(selected_review) == 21

# ------------------------------------------------------------
# D. VALIDATE LONG TEXT DECISIONS
# ------------------------------------------------------------

assert long_review["review_decision"].eq(
    "KEEP_FULL"
).all(), "Masih ada teks panjang yang belum diputuskan."

# ------------------------------------------------------------
# E. BUILD VERIFIED DATASET
# ------------------------------------------------------------

final_df = dup_df.copy()
final_df["id"] = final_df["id"].astype("string")

selected_ids = set(selected_review["id"])

final_df["model_selection_status"] = "SELECTED_UNIQUE"

duplicate_mask = final_df["is_duplicate_in_case"]

final_df.loc[
    duplicate_mask,
    "model_selection_status"
] = "GROUP_MEMBER_NOT_SELECTED"

final_df.loc[
    final_df["id"].isin(selected_ids),
    "model_selection_status"
] = "SELECTED_REPRESENTATIVE"

# Semua teks panjang tetap penuh.
long_ids = set(long_review["id"])

final_df["long_text_review_status"] = "NOT_LONG_TEXT"

final_df.loc[
    final_df["id"].isin(long_ids),
    "long_text_review_status"
] = "KEEP_FULL"

final_df["eligible_for_model_candidate"] = (
    final_df["model_selection_status"].isin([
        "SELECTED_UNIQUE",
        "SELECTED_REPRESENTATIVE"
    ])
)

# ------------------------------------------------------------
# F. FINAL ASSERTIONS
# ------------------------------------------------------------

assert len(final_df) == 381
assert final_df["id"].is_unique

assert (
    final_df["model_selection_status"]
    .eq("SELECTED_REPRESENTATIVE")
    .sum()
) == 21

assert (
    final_df["model_selection_status"]
    .eq("GROUP_MEMBER_NOT_SELECTED")
    .sum()
) == 33

assert (
    final_df["eligible_for_model_candidate"]
    .sum()
) == 348

assert (
    final_df["long_text_review_status"]
    .eq("KEEP_FULL")
    .sum()
) == 12

assert final_df["text_original"].equals(
    dup_df["text_original"]
)

assert final_df["text_linguistic"].equals(
    dup_df["text_linguistic"]
)

# ------------------------------------------------------------
# G. SAVE DERIVED AUDIT OUTPUTS
# ------------------------------------------------------------

recon_dir = OUTPUT_DIR / "reconciliation"
recon_dir.mkdir(parents=True, exist_ok=True)

full_path = recon_dir / "part8_content_reconciled.csv"
model_path = recon_dir / "part8_model_candidates.csv"
summary_path = recon_dir / "part8_summary.json"

for path in [full_path, model_path, summary_path]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah ada: {path}"
        )

export_cols = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "duplicate_group_id",
    "duplicate_group_size",
    "model_selection_status",
    "long_text_review_status",
    "eligible_for_model_candidate"
]

final_df[export_cols].to_csv(
    full_path,
    index=False,
    encoding="utf-8-sig"
)

final_df.loc[
    final_df["eligible_for_model_candidate"],
    export_cols
].to_csv(
    model_path,
    index=False,
    encoding="utf-8-sig"
)

summary_08 = {
    "total_tweets": 381,
    "duplicate_groups": 21,
    "selected_representatives": 21,
    "duplicate_members_not_selected": 33,
    "unique_nonduplicate_tweets": 327,
    "model_candidates": 348,
    "long_texts_keep_full": 12,
    "rows_deleted_from_master": 0,
    "status": "PASS"
}

summary_path.write_text(
    json.dumps(
        summary_08,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# H. REPORT
# ------------------------------------------------------------

print("=" * 64)
print("08 — CONTENT REVIEW RECONCILIATION")
print("=" * 64)

for key, value in summary_08.items():
    print(f"{key}: {value}")

print("\nDataset distribution:")
display(
    final_df.groupby(
        ["case", "model_selection_status"]
    ).size().unstack(fill_value=0)
)

print("\nOutputs:")
print(full_path)
print(model_path)
print(summary_path)

print("\nCELL 08 CONTENT RECONCILIATION: PASS")


08 — CONTENT REVIEW RECONCILIATION
total_tweets: 381
duplicate_groups: 21
selected_representatives: 21
duplicate_members_not_selected: 33
unique_nonduplicate_tweets: 327
model_candidates: 348
long_texts_keep_full: 12
rows_deleted_from_master: 0
status: PASS

Dataset distribution:


model_selection_status,GROUP_MEMBER_NOT_SELECTED,SELECTED_REPRESENTATIVE,SELECTED_UNIQUE
case,,,
lombok_2018,13,9,92
palu_2018,13,9,96
sumatra_2025,7,3,139



Outputs:
C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\reconciliation\part8_content_reconciled.csv
C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\reconciliation\part8_model_candidates.csv
C:\Penelitian Sentimen Bencana Nasional\02_Cleaning\outputs\reconciliation\part8_summary.json

CELL 08 CONTENT RECONCILIATION: PASS


In [12]:
# 09 — FINAL CLEANING INTEGRITY & LABELING HANDOFF
# OFFLINE ONLY | NO DATA DELETION | NO AUTO LABELING
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from openpyxl.styles import Font, Alignment

assert len(final_df) == 381
assert final_df["id"].astype(str).is_unique

handoff = final_df.copy()
handoff["id"] = handoff["id"].astype("string")

# A. Validasi status final
expected_status = {
    "SELECTED_UNIQUE": 327,
    "SELECTED_REPRESENTATIVE": 21,
    "GROUP_MEMBER_NOT_SELECTED": 33,
}

counts = handoff["model_selection_status"].value_counts()

for status, expected in expected_status.items():
    assert int(counts.get(status, 0)) == expected, (
        f"Jumlah {status} tidak sesuai."
    )

assert handoff["text_original"].astype(
    "string"
).str.strip().ne("").all()

assert handoff["text_linguistic"].astype(
    "string"
).str.strip().ne("").all()

# Validasi hash teks global.
handoff["final_text_hash"] = handoff[
    "text_linguistic"
].map(
    lambda value: hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()
)

model_mask = handoff[
    "eligible_for_model_candidate"
].astype(bool)

model_df = handoff.loc[model_mask].copy()

assert len(model_df) == 348

# Tidak boleh ada teks identik dalam kandidat model.
assert model_df["final_text_hash"].is_unique, (
    "Masih ada teks identik pada kandidat modeling."
)

assert set(model_df["id"]).issubset(
    set(handoff["id"])
)

# B. Dataset untuk anotasi
# Kolom sentimen sengaja dikosongkan.
columns = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "model_selection_status",
    "duplicate_group_id",
    "duplicate_group_size",
    "long_text_review_status",
    "eligible_for_model_candidate",
    "final_text_hash",
]

labeling_all = handoff[columns].copy()
labeling_model = model_df[columns].copy()

for frame in (labeling_all, labeling_model):
    frame["sentiment_label"] = ""
    frame["label_notes"] = ""
    frame["annotator"] = ""
    frame["labeled_at"] = ""

# C. Export ke folder khusus
LABELING_DIR = ROOT / "03_Labeling" / "inputs"
LABELING_DIR.mkdir(parents=True, exist_ok=True)

all_csv = LABELING_DIR / "labeling_all_381.csv"
model_csv = LABELING_DIR / "labeling_candidates_348.csv"
excel_path = LABELING_DIR / "labeling_template.xlsx"
manifest_path = LABELING_DIR / "cleaning_handoff_manifest.json"

for path in (all_csv, model_csv, excel_path, manifest_path):
    if path.exists():
        raise FileExistsError(
            f"Output sudah ada: {path}"
        )

labeling_all.to_csv(
    all_csv, index=False, encoding="utf-8-sig"
)

labeling_model.to_csv(
    model_csv, index=False, encoding="utf-8-sig"
)

# D. Excel template.
# Simpan ID sebagai teks dan cegah formula injection
# tanpa menambahkan apostrof pada isi tweet.
with pd.ExcelWriter(
    excel_path, engine="openpyxl"
) as writer:

    labeling_all.to_excel(
        writer,
        sheet_name="ALL_381",
        index=False
    )

    labeling_model.to_excel(
        writer,
        sheet_name="MODEL_348",
        index=False
    )

    for ws in writer.sheets.values():
        ws.freeze_panes = "C2"
        ws.auto_filter.ref = ws.dimensions

        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(wrap_text=True)

        for col in ws.columns:
            header = str(col[0].value)
            letter = col[0].column_letter

            ws.column_dimensions[letter].width = (
                70 if header.startswith("text_") else 25
            )

            for cell in col[1:]:
                if isinstance(cell.value, str):
                    # Paksa string literal, bukan formula.
                    cell.data_type = "s"

                if header == "id":
                    cell.number_format = "@"

# E. Manifest
manifest = {
    "source_stage": "02_Cleaning",
    "target_stage": "03_Labeling",
    "total_eligible_tweets": 381,
    "model_candidates": 348,
    "duplicate_members_not_selected": 33,
    "long_texts_preserved": 12,
    "labels_assigned": 0,
    "status": "READY_FOR_LABELING"
}

manifest_path.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

print("=" * 62)
print("09 — FINAL CLEANING HANDOFF")
print("=" * 62)

for key, value in manifest.items():
    print(f"{key}: {value}")

print("\nOutputs:")
print(all_csv)
print(model_csv)
print(excel_path)
print(manifest_path)

print("\nCELL 09 HANDOFF: PASS")
print("No automatic sentiment labels assigned.")
print("All original tweets retained.")


09 — FINAL CLEANING HANDOFF
source_stage: 02_Cleaning
target_stage: 03_Labeling
total_eligible_tweets: 381
model_candidates: 348
duplicate_members_not_selected: 33
long_texts_preserved: 12
labels_assigned: 0
status: READY_FOR_LABELING

Outputs:
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\inputs\labeling_all_381.csv
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\inputs\labeling_candidates_348.csv
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\inputs\labeling_template.xlsx
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\inputs\cleaning_handoff_manifest.json

CELL 09 HANDOFF: PASS
No automatic sentiment labels assigned.
All original tweets retained.
